# Домашнее задание 2. Полносвязные нейронные сети

Реализуйте полносвязную нейронную сеть на NumPy и исследуйте её обучение
на задаче двумерной классификации. Работа состоит из двух частей:
реализации и проверки компонентов, затем сравнения моделей и режимов обучения.


**Фамилия и имя:**  
**Группа:**


## Предварительные сведения

Предполагается знание матричного дифференцирования, обратного распространения
ошибки, softmax, кросс-энтропии и стохастического градиентного спуска,
а также опыт работы с массивами NumPy.

Слои, аналитические градиенты, оптимизатор и цикл обучения необходимо реализовать
самостоятельно. Автоматическое дифференцирование и готовые реализации
нейросетевых моделей использовать нельзя. Численное дифференцирование
допускается только при проверке градиентов. Генераторы данных, средства
предобработки, работы с таблицами и визуализации можно брать из библиотек.

Для выполнения нужны Python 3.10+, NumPy, pandas, matplotlib и scikit-learn.
Все вещественные вычисления проводите в `float64`, метки классов храните
как целые числа. Данные генерируются в ноутбуке; достаточно CPU.


## Порядок работы и формат сдачи

Заполните пропуски `TODO` и ячейки для решения. Сигнатуры методов и предоставленные
проверки сохраняйте. Проверки выводят `OK` при успешном выполнении
и `НЕ РЕАЛИЗОВАНО`, если соответствующий компонент ещё не написан;
ошибка проверки прерывает выполнение.

Сдайте один `.ipynb` с кодом, сохранёнными результатами вычислений и ответами
непосредственно под соответствующими заданиями. Отдельный отчёт не требуется.
Таблицы должны содержать настройки экспериментов, графики — подписи осей
и легенды. Обосновывайте выводы полученными числами и графиками.

Перед сдачей перезапустите ядро, выполните все ячейки по порядку и сохраните
ноутбук. Решение должно воспроизводиться без внешних файлов с кодом.


In [ ]:
# При необходимости выполните, затем перезапустите ядро.
# %pip install numpy pandas matplotlib scikit-learn

In [ ]:
import copy
import sys
from collections.abc import Callable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.datasets import make_blobs, make_circles
from sklearn.model_selection import train_test_split

SEED = 42
SHUFFLE_SEED = 1042
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.figsize": (7, 4.5), "figure.dpi": 100})
print(
    f"Python {sys.version.split()[0]} | NumPy {np.__version__} | "
    f"pandas {pd.__version__} | matplotlib {matplotlib.__version__} | "
    f"scikit-learn {sklearn.__version__}"
)

## Задание 1. Реализация нейронной сети

Основная архитектура: `Linear(2, h) → ReLU → Linear(h, 2)`.
Для линейного классификатора используется `Linear(2, 2)`.
Выход модели — логиты. Реализуемые компоненты должны поддерживать
произвольные положительные размерности входа, скрытого слоя и выхода.

**Интерфейс.** Объекты расположены по строкам: вход слоя имеет форму
`(batch_size, in_features)`. Метод `forward` возвращает выход слоя,
`backward` принимает градиент по выходу и возвращает градиент по входу
для последнего прямого прохода. Градиенты параметров записываются в `dW`
и `db` заново при каждом вызове `backward`.

Методы не должны изменять входные массивы. Параметры обновляет только
оптимизатор; `backward` оставляет их неизменными. Порядок массивов
в `parameters()` и `gradients()` должен совпадать. Каждый экземпляр слоя
используется в сети один раз; общих параметров между слоями нет.
Усреднение по объектам выполняется в функции потерь.


In [ ]:
class Module:
    """Интерфейс слоя."""

    def forward(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def parameters(self) -> list[np.ndarray]:
        return []

    def gradients(self) -> list[np.ndarray]:
        return []

    def zero_grad(self) -> None:
        for gradient in self.gradients():
            gradient.fill(0.0)

### 1.1. Линейный слой и ReLU

Реализуйте `forward` и `backward` в классах `Linear` и `ReLU`.
Линейный слой задаётся преобразованием

$$
Y = XW + b,
$$

где `W.shape == (in_features, out_features)`, а
`b.shape == (out_features,)`. Для ReLU примите производную в нуле равной нулю.
Используйте векторизованные операции без циклов по объектам и нейронам.
Предоставленную инициализацию параметров сохраняйте.

Выполните проверки линейного слоя и ReLU.


In [ ]:
class Linear(Module):
    def __init__(
        self, in_features: int, out_features: int, rng: np.random.Generator
    ):
        if min(in_features, out_features) < 1:
            raise ValueError("Размеры слоя должны быть положительными.")
        scale = 1.0 / np.sqrt(in_features)
        self.W = rng.normal(0.0, scale, size=(in_features, out_features))
        self.b = rng.uniform(-scale, scale, size=out_features)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._input = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("Linear.forward")

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("Linear.backward")

    def parameters(self) -> list[np.ndarray]:
        return [self.W, self.b]

    def gradients(self) -> list[np.ndarray]:
        return [self.dW, self.db]


class ReLU(Module):
    def __init__(self):
        self._input = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("ReLU.forward")

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("ReLU.backward")

In [ ]:
# Проверки компонентов и цикла обучения.


def numerical_gradient(
    objective: Callable[[], float], array: np.ndarray, eps: float = 1e-6
) -> np.ndarray:
    """objective читает array; значения array временно меняются и восстанавливаются."""
    if array.dtype != np.float64:
        raise TypeError("Для численной проверки используйте float64.")
    result = np.empty_like(array)
    for idx in np.ndindex(array.shape):
        original = float(array[idx])
        try:
            array[idx] = original + eps
            plus = float(objective())
            array[idx] = original - eps
            minus = float(objective())
        finally:
            array[idx] = original
        result[idx] = (plus - minus) / (2.0 * eps)
    return result


def compare_gradients(analytic: np.ndarray, numeric: np.ndarray) -> float:
    assert analytic.shape == numeric.shape, "Не совпадают формы градиентов."
    assert np.all(np.isfinite(analytic)), "В градиенте есть NaN/inf."
    error = np.linalg.norm(analytic - numeric) / max(
        1e-12, np.linalg.norm(analytic) + np.linalg.norm(numeric)
    )
    np.testing.assert_allclose(analytic, numeric, rtol=1e-4, atol=1e-7)
    assert error < 1e-5, f"Слишком большая относительная ошибка: {error:.3g}"
    return float(error)


def run_checks(checks: dict) -> pd.DataFrame:
    rows = []
    for name, check in checks.items():
        try:
            details = check()
        except NotImplementedError as exc:
            rows.append(
                {
                    "проверка": name,
                    "статус": "НЕ РЕАЛИЗОВАНО",
                    "детали": str(exc),
                }
            )
        else:
            rows.append({"проверка": name, "статус": "OK", "детали": details})
    result = pd.DataFrame(rows)
    with pd.option_context("display.max_colwidth", None):
        display(result)
    return result


def check_linear():
    rng = np.random.default_rng(101)
    layer = Linear(2, 3, rng)
    X = rng.normal(size=(4, 2))
    G = rng.normal(size=(4, 3))
    X_before, G_before = X.copy(), G.copy()
    weights_before = [p.copy() for p in layer.parameters()]
    out = layer.forward(X)
    np.testing.assert_allclose(out, X @ layer.W + layer.b)
    dx = layer.backward(G).copy()
    dw, db = layer.dW.copy(), layer.db.copy()
    assert layer.dW.shape == layer.W.shape and layer.db.shape == layer.b.shape
    np.testing.assert_array_equal(X, X_before)
    np.testing.assert_array_equal(G, G_before)
    for p, old in zip(layer.parameters(), weights_before):
        np.testing.assert_array_equal(p, old)

    def objective():
        return np.sum(layer.forward(X) * G)

    errors = {
        "dX": compare_gradients(dx, numerical_gradient(objective, X)),
        "dW": compare_gradients(dw, numerical_gradient(objective, layer.W)),
        "db": compare_gradients(db, numerical_gradient(objective, layer.b)),
    }
    # Повторный backward не должен накапливать градиенты предыдущего батча.
    layer.forward(X)
    layer.backward(G)
    np.testing.assert_allclose(layer.dW, dw)
    np.testing.assert_allclose(layer.db, db)
    layer.zero_grad()
    assert all(np.all(g == 0.0) for g in layer.gradients())
    return errors


def check_relu():
    layer = ReLU()
    X = np.array([[-2.0, 0.0, 3.0], [1.0, -4.0, 0.0]])
    G = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    X_before, G_before = X.copy(), G.copy()
    np.testing.assert_array_equal(
        layer.forward(X), [[0.0, 0.0, 3.0], [1.0, 0.0, 0.0]]
    )
    np.testing.assert_array_equal(
        layer.backward(G), [[0.0, 0.0, 3.0], [4.0, 0.0, 0.0]]
    )
    np.testing.assert_array_equal(X, X_before)
    np.testing.assert_array_equal(G, G_before)
    # Для численной проверки отдельно берём входы без точек излома.
    X = np.array([[-0.7, 1.3], [0.8, -2.1]])
    G = np.array([[0.3, -0.9], [1.4, 0.5]])
    layer.forward(X)
    analytic = layer.backward(G).copy()
    numeric = numerical_gradient(lambda: np.sum(layer.forward(X) * G), X)
    return {"dX вне нуля": compare_gradients(analytic, numeric)}


def check_loss():
    rng = np.random.default_rng(102)
    logits = rng.normal(size=(4, 3))
    y = np.array([0, 2, 1, 0])
    before, y_before = logits.copy(), y.copy()
    loss, gradient = softmax_cross_entropy(logits, y)
    assert np.ndim(loss) == 0 and gradient.shape == logits.shape
    # Независимый способ проверки значений через ufunc NumPy.
    expected = np.mean(
        np.logaddexp.reduce(logits, axis=1) - logits[np.arange(4), y]
    )
    np.testing.assert_allclose(loss, expected, rtol=1e-12, atol=1e-12)
    np.testing.assert_array_equal(logits, before)
    np.testing.assert_array_equal(y, y_before)
    numeric = numerical_gradient(
        lambda: softmax_cross_entropy(logits, y)[0], logits
    )
    return {"dLogits": compare_gradients(gradient.copy(), numeric)}


def check_loss_properties():
    logits = np.array(
        [
            [1000.0, -1000.0, 0.0],
            [-1000.0, 1000.0, 0.0],
            [1000.0, 1000.0, 1000.0],
        ]
    )
    y = np.array([1, 1, 2])
    # Underflow exp(-2000) -> 0 допустим; переполнение и NaN — нет.
    with np.errstate(
        over="raise", invalid="raise", divide="raise", under="ignore"
    ):
        loss, grad = softmax_cross_entropy(logits, y)
    assert np.isfinite(loss) and np.all(np.isfinite(grad))
    np.testing.assert_allclose(loss, (2000.0 + np.log(3.0)) / 3.0)
    shifts = np.array([[123.0], [-321.0], [1000.0]])
    shifted_loss, shifted_grad = softmax_cross_entropy(logits + shifts, y)
    np.testing.assert_allclose(shifted_loss, loss, rtol=1e-12, atol=1e-10)
    np.testing.assert_allclose(shifted_grad, grad, rtol=1e-12, atol=1e-12)
    # Удвоение батча не меняет среднюю потерю; вклады двух копий складываются.
    duplicate_loss, duplicate_grad = softmax_cross_entropy(
        np.tile(logits, (2, 1)), np.tile(y, 2)
    )
    np.testing.assert_allclose(duplicate_loss, loss)
    np.testing.assert_allclose(duplicate_grad[:3] + duplicate_grad[3:], grad)
    np.testing.assert_allclose(grad.sum(axis=1), 0.0, atol=1e-12)
    return "Большие логиты, сдвиг строк, среднее по батчу: OK"


def check_network():
    rng = np.random.default_rng(103)
    first, last = Linear(2, 4, rng), Linear(4, 3, rng)
    first.b[:] = [0.7, -0.8, 0.9, 0.6]
    network = Sequential(first, ReLU(), last)
    X = rng.normal(size=(3, 2))
    y = np.array([0, 1, 2])
    assert np.min(np.abs(X @ first.W + first.b)) > 1e-4
    logits = network.forward(X)
    assert logits.shape == (3, 3)
    reference = np.maximum(X @ first.W + first.b, 0.0) @ last.W + last.b
    np.testing.assert_allclose(logits, reference)
    params = network.parameters()
    assert len(params) == 4 and all(
        p is q for p, q in zip(params, [first.W, first.b, last.W, last.b])
    )
    before = [p.copy() for p in params]
    _, dlogits = softmax_cross_entropy(logits, y)
    dlogits_before = dlogits.copy()
    dx = network.backward(dlogits).copy()
    gradients = [g.copy() for g in network.gradients()]
    np.testing.assert_array_equal(dlogits, dlogits_before)
    for p, old in zip(params, before):
        np.testing.assert_array_equal(p, old)

    def objective():
        return softmax_cross_entropy(network.forward(X), y)[0]

    errors = {
        name: compare_gradients(g, numerical_gradient(objective, p))
        for name, p, g in zip(["W1", "b1", "W2", "b2"], params, gradients)
    }
    errors["X"] = compare_gradients(dx, numerical_gradient(objective, X))
    # Изменение размера батча не должно использовать старый сохранённый вход.
    out = network.forward(X[:2])
    assert out.shape == (2, 3)
    network.backward(np.ones_like(out))
    assert first.dW.shape == first.W.shape
    return errors


def check_sgd():
    layer = Linear(2, 3, np.random.default_rng(104))
    layer.dW[:] = 0.2
    layer.db[:] = [-0.1, 0.3, 0.4]
    optimizer = SGD(layer, lr=0.05)
    refs = layer.parameters()
    before = [p.copy() for p in refs]
    grad_before = [g.copy() for g in layer.gradients()]
    optimizer.step()
    for p, ref, old, grad in zip(
        layer.parameters(), refs, before, grad_before
    ):
        assert p is ref, (
            "Обновляйте значения массива, не заменяйте параметр новым объектом."
        )
        np.testing.assert_allclose(p, old - 0.05 * grad)
    # Оптимизатор должен читать текущие градиенты, не копию из __init__.
    layer.dW = np.full_like(layer.W, -0.3)
    layer.db = np.full_like(layer.b, 0.2)
    before = [p.copy() for p in layer.parameters()]
    optimizer.step()
    for p, old, grad in zip(layer.parameters(), before, layer.gradients()):
        np.testing.assert_allclose(p, old - 0.05 * grad)
    return "Правильный знак, текущие градиенты, обновление на месте: OK"


def _training_fixture():
    X = np.array(
        [[-1.0, -0.5], [-0.7, -1.2], [0.8, 0.7], [1.3, 0.9], [1.0, 1.4]]
    )
    y = np.array([0, 0, 1, 1, 1])
    V = np.array([[-0.9, -0.8], [0.8, 1.1], [1.5, 0.3]])
    vy = np.array([0, 1, 1])
    return X, y, V, vy


def check_fit():
    X, y, V, vy = _training_fixture()
    untouched = [a.copy() for a in (X, y, V, vy)]
    seen = []

    class RecordingLinear(Linear):
        def backward(self, grad_output):
            seen.append(self._input.copy())
            return super().backward(grad_output)

    model = Sequential(RecordingLinear(2, 2, np.random.default_rng(105)))
    initial = evaluate(model, X, y)
    history = fit(
        model, X, y, V, vy, epochs=3, lr=0.1, batch_size=2, shuffle_seed=19
    )
    required = {"epoch", "train_loss", "valid_loss", "train_acc", "valid_acc"}
    assert isinstance(history, pd.DataFrame) and required.issubset(
        history.columns
    )
    np.testing.assert_array_equal(history["epoch"], np.arange(4))
    assert np.all(np.isfinite(history[list(required)].to_numpy()))
    np.testing.assert_allclose(history.iloc[0]["train_loss"], initial["loss"])
    np.testing.assert_allclose(history.iloc[0]["train_acc"], initial["acc"])
    assert [len(a) for a in seen] == [2, 2, 1] * 3, (
        "Проверьте число и размеры батчей."
    )
    for start in (0, 3, 6):
        collected = np.vstack(seen[start : start + 3])
        assert sorted(map(tuple, collected)) == sorted(map(tuple, X)), (
            "За эпоху нужны все объекты ровно один раз."
        )
    for part, xx, yy in [("train", X, y), ("valid", V, vy)]:
        metrics = evaluate(model, xx, yy)
        for metric in ("loss", "acc"):
            np.testing.assert_allclose(
                history.iloc[-1][f"{part}_{metric}"], metrics[metric]
            )
    assert history.iloc[-1]["train_loss"] < initial["loss"]
    replica = Sequential(Linear(2, 2, np.random.default_rng(105)))
    again = fit(
        replica, X, y, V, vy, epochs=3, lr=0.1, batch_size=2, shuffle_seed=19
    )
    np.testing.assert_allclose(
        history[sorted(required)], again[sorted(required)]
    )
    for actual, old in zip((X, y, V, vy), untouched):
        np.testing.assert_array_equal(actual, old)
    # Корректность при batch_size больше всей выборки.
    fit(replica, X, y, V, vy, epochs=1, lr=0.1, batch_size=99, shuffle_seed=19)
    return "Последний батч, все объекты, история, воспроизводимость: OK"


def check_validation_isolation():
    X, y, V, vy = _training_fixture()
    first = make_linear(seed=106)
    second = make_linear(seed=106)
    fit(first, X, y, V, vy, epochs=3, lr=0.1, batch_size=2, shuffle_seed=19)
    fit(
        second,
        X,
        y,
        V + 2.0,
        1 - vy,
        epochs=3,
        lr=0.1,
        batch_size=2,
        shuffle_seed=19,
    )
    for p, q in zip(first.parameters(), second.parameters()):
        np.testing.assert_allclose(p, q, rtol=0.0, atol=1e-12)
    return "Изменение validation не изменяет обученные параметры: OK"

In [ ]:
checks_11 = run_checks({"Linear": check_linear, "ReLU": check_relu})

### 1.2. Композиция слоёв и функция потерь

Реализуйте `Sequential.forward` и `Sequential.backward` для последовательной
композиции слоёв.

Реализуйте `softmax_cross_entropy(logits, y)`. Функция принимает логиты
формы `(B, K)` и целочисленные метки формы `(B,)` со значениями от `0`
до `K - 1`. Она должна возвращать среднюю кросс-энтропию и её аналитический
градиент по логитам той же формы, что и вход:

$$
L = -\frac{1}{B}\sum_{i=1}^{B}
\log\bigl(\operatorname{softmax}(z_i)_{y_i}\bigr).
$$

Обеспечьте численную устойчивость, в том числе для логитов порядка `±1000`.
Обрезание потерь и замена нечисловых значений не допускаются.
Выполните проверки функции потерь и композиции слоёв.


In [ ]:
class Sequential(Module):
    def __init__(self, *layers: Module):
        if not layers:
            raise ValueError("Сеть должна содержать хотя бы один слой.")
        self.layers = list(layers)

    def forward(self, X: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("Sequential.forward")

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError("Sequential.backward")

    def parameters(self) -> list[np.ndarray]:
        return [p for layer in self.layers for p in layer.parameters()]

    def gradients(self) -> list[np.ndarray]:
        return [g for layer in self.layers for g in layer.gradients()]


def softmax_cross_entropy(logits: np.ndarray, y: np.ndarray):
    """Возвращает (float: средняя потеря, ndarray: градиент по логитам)."""
    if logits.ndim != 2 or len(logits) == 0 or y.shape != (len(logits),):
        raise ValueError("Ожидаются непустые logits (B, K) и y (B,).")
    if not np.issubdtype(y.dtype, np.integer):
        raise TypeError("Метки должны быть целыми числами.")
    if np.any(y < 0) or np.any(y >= logits.shape[1]):
        raise ValueError("Метка вне диапазона классов.")
    # TODO
    raise NotImplementedError("softmax_cross_entropy")

In [ ]:
checks_12 = run_checks(
    {
        "Cross-entropy: значения и градиент": check_loss,
        "Cross-entropy: устойчивость и нормировка": check_loss_properties,
        "Sequential: градиенты всей цепочки": check_network,
    }
)

### 1.3. Оптимизатор и цикл обучения

Реализуйте `SGD.step` без инерции, регуляризации и адаптации шага.
Обновляйте массивы параметров на месте, используя текущие градиенты модели.

Реализуйте `fit`: функция обучает переданную модель мини-батчами и возвращает
`pandas.DataFrame` со столбцами `epoch`, `train_loss`, `valid_loss`,
`train_acc`, `valid_acc`. В таблице должно быть `epochs + 1` строк:
исходное состояние (`epoch=0`) и состояние после каждой эпохи.
Метрики вычисляются на полных выборках с параметрами соответствующей эпохи.
Используйте предоставленную функцию `evaluate`.

За эпоху каждый обучающий объект должен участвовать ровно в одном обновлении,
включая объекты последнего неполного батча. Порядок объектов перемешивается
каждую эпоху с помощью предоставленного генератора `rng`.
Валидационная выборка не участвует в обновлении параметров.
При нечисловой потере завершайте запуск с `FloatingPointError`.

Выполните проверки оптимизатора и цикла обучения.


In [ ]:
def make_linear(seed: int = SEED) -> Sequential:
    return Sequential(Linear(2, 2, np.random.default_rng(seed)))


def make_mlp(
    hidden: int = 16, activation: str = "relu", seed: int = SEED
) -> Sequential:
    rng = np.random.default_rng(seed)
    if activation == "relu":
        layer = ReLU()
    elif activation == "step":
        layer = Step()  # Класс определён в пункте 2.4.
    else:
        raise ValueError("Поддерживаются 'relu' и 'step'.")
    return Sequential(Linear(2, hidden, rng), layer, Linear(hidden, 2, rng))


def evaluate(model: Module, X: np.ndarray, y: np.ndarray) -> dict:
    logits = model.forward(X)
    loss, _ = softmax_cross_entropy(logits, y)
    if not np.isfinite(loss):
        raise FloatingPointError(
            "Нечисловая потеря: проверьте вычисления и шаг обучения."
        )
    return {
        "loss": float(loss),
        "acc": float(np.mean(logits.argmax(axis=1) == y)),
    }

In [ ]:
class SGD:
    def __init__(self, model: Module, lr: float = 0.1):
        if not np.isfinite(lr) or lr <= 0:
            raise ValueError("lr должен быть конечным положительным числом.")
        self.model = model
        self.lr = lr

    def step(self) -> None:
        # TODO
        raise NotImplementedError("SGD.step")


def fit(
    model: Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    *,
    epochs: int = 200,
    lr: float = 0.1,
    batch_size: int = 32,
    shuffle_seed: int = SHUFFLE_SEED,
) -> pd.DataFrame:
    """Обучает model; возвращает историю с epochs + 1 строками, включая epoch=0."""
    if epochs < 1 or batch_size < 1:
        raise ValueError("epochs и batch_size должны быть положительными.")
    if len(X_train) != len(y_train) or len(X_valid) != len(y_valid):
        raise ValueError("Количество объектов и меток должно совпадать.")
    rng = np.random.default_rng(shuffle_seed)
    optimizer = SGD(model, lr)
    # TODO
    raise NotImplementedError("fit")

In [ ]:
checks_13 = run_checks(
    {
        "SGD: один шаг": check_sgd,
        "fit: батчи, метрики и воспроизводимость": check_fit,
        "fit: validation не меняет обучение": check_validation_isolation,
    }
)

### 1.4. Проверка корректности

1. Убедитесь, что все предоставленные проверки завершились со статусом `OK`.
   Представьте таблицу относительных ошибок аналитических градиентов.
   Добавьте собственный тест случая, не покрытого предоставленными проверками,
   и объясните его назначение.
2. Обучите сеть `2 → 8 → 2` с ReLU на 16 объектах, заданных ниже.
   Добейтесь снижения кросс-энтропии и правильной классификации всех объектов.
   Представьте начальные и конечные значения потерь и accuracy, а также
   нормы изменения матриц весов обоих слоёв. Здесь и далее для матриц
   используйте норму Фробениуса, для векторов — евклидову норму.

В этом пункте отложенная выборка не используется. При вызове `fit` передайте
`sanity_X, sanity_y` также в аргументы валидационной выборки.
Укажите, какие свойства реализации подтверждают проведённые проверки
и какие ограничения у них остаются.


In [ ]:
_sanity_rng = np.random.default_rng(7)
sanity_X = np.vstack(
    [
        _sanity_rng.normal((-1.5, -1.0), 0.18, size=(8, 2)),
        _sanity_rng.normal((1.5, 1.0), 0.18, size=(8, 2)),
    ]
).astype(np.float64)
sanity_y = np.repeat([0, 1], 8).astype(np.int64)
del _sanity_rng

In [ ]:
# Решение пункта 1.4.


**Ответ:**


## Задание 2. Исследование обучения

Используйте реализацию из задания 1. Даны два набора по 600 объектов:
гауссовские облака (`clouds`) и вложенные окружности (`circles`).
Стратифицированное разбиение на обучающую, валидационную и тестовую выборки
выполнено в пропорции 60 / 20 / 20. Параметры генерации и разбиение фиксированы.

**Протокол экспериментов.** Параметры предобработки и модели оценивайте только
по `train`, конфигурацию выбирайте по `validation`. К `test` обращайтесь
только в пункте 2.5. Во всех сравнениях сохраняйте выбранную предобработку,
число эпох и размер батча; если не оговорено иное, меняйте только исследуемый
параметр. Каждый запуск начинается с исходной инициализации.

До сравнений в пунктах 2.2–2.4 сформулируйте гипотезы. Сохраняйте настройки,
обученные модели и истории всех запусков, включая неудачные. Для завершившихся
запусков сравнивайте метрики последней эпохи. Полностью совпадающие запуски
можно использовать повторно. Заданного порога accuracy на этих данных нет.


In [ ]:
def split_dataset(X: np.ndarray, y: np.ndarray) -> dict:
    idx_train, idx_hold = train_test_split(
        np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED
    )
    idx_valid, idx_test = train_test_split(
        idx_hold, test_size=0.5, stratify=y[idx_hold], random_state=SEED
    )
    assert not (
        set(idx_train) & set(idx_valid)
        or set(idx_train) & set(idx_test)
        or set(idx_valid) & set(idx_test)
    )
    result = {}
    for name, idx in [
        ("train", idx_train),
        ("valid", idx_valid),
        ("test", idx_test),
    ]:
        result[f"X_{name}"] = np.asarray(X[idx], dtype=np.float64)
        result[f"y_{name}"] = np.asarray(y[idx], dtype=np.int64)
    return result


data = {
    "clouds": split_dataset(
        *make_blobs(
            n_samples=600,
            centers=[[-2.0, -1.0], [2.0, 1.0]],
            cluster_std=0.9,
            random_state=SEED,
        )
    ),
    "circles": split_dataset(
        *make_circles(
            n_samples=600,
            factor=0.5,
            noise=0.12,
            random_state=SEED,
        )
    ),
}
for name, dataset in data.items():
    print(
        name,
        {
            part: len(dataset[f"y_{part}"])
            for part in ("train", "valid", "test")
        },
    )

### 2.1. Данные и предобработка

Для обучающей части каждого набора представьте доли классов, диапазоны
признаков, число пропусков и дубликатов, а также диаграмму рассеяния
с обозначением классов.

Выберите и обоснуйте предобработку. Подготовленные данные сохраните
в `prepared`, сохранив структуру словаря `data`. Исходный `data` не изменяйте.
Если предобработка не требуется, сохраните в `prepared` копию исходных данных.

Для каждого набора оцените на validation accuracy классификатора,
предсказывающего наиболее частый класс train. При равенстве частот выберите
класс `0`. Сопоставьте геометрию наборов с возможностями линейной модели.


In [ ]:
# Решение пункта 2.1.


**Ответ:**


Функция `plot_boundary` строит разделяющую границу модели, обучающие объекты
и ошибки на validation. Аргументы должны быть представлены в той же шкале,
в которой обучалась модель.


In [ ]:
def plot_boundary(
    model: Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    title: str,
):
    low, high = X_train.min(axis=0) - 0.4, X_train.max(axis=0) + 0.4
    xx, yy = np.meshgrid(
        np.linspace(low[0], high[0], 160), np.linspace(low[1], high[1], 160)
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    grid_logits = model.forward(grid)
    score = (grid_logits[:, 1] - grid_logits[:, 0]).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6, 5))
    if score.min() < 0 < score.max():
        ax.contour(xx, yy, score, levels=[0.0], linewidths=1.6)
    for label, marker in [(0, "o"), (1, "s")]:
        mask = y_train == label
        ax.scatter(
            X_train[mask, 0],
            X_train[mask, 1],
            marker=marker,
            s=18,
            alpha=0.5,
            label=f"train: класс {label}",
        )
    wrong = model.forward(X_valid).argmax(axis=1) != y_valid
    if np.any(wrong):
        ax.scatter(
            X_valid[wrong, 0],
            X_valid[wrong, 1],
            marker="x",
            s=42,
            label="ошибка validation",
        )
    ax.set(
        title=title,
        xlabel="x1 (рабочая шкала)",
        ylabel="x2 (рабочая шкала)",
        xlim=(low[0], high[0]),
        ylim=(low[1], high[1]),
    )
    ax.legend()
    fig.tight_layout()
    plt.show()

### 2.2. Линейная модель и сеть со скрытым слоем

На каждом наборе обучите линейный классификатор `make_linear()` и сеть
`make_mlp(hidden=16)` с ReLU — всего четыре модели.
Используйте `seed=SEED`, `shuffle_seed=SHUFFLE_SEED` и единый режим обучения:
`epochs=200`, `lr=0.1`, `batch_size=32`.

Представьте таблицу train/validation loss и accuracy для четырёх моделей.
Для окружностей постройте кривые train/validation loss и разделяющие границы
обеих моделей.

Объясните различия результатов с учётом геометрии данных, выразительности
моделей и хода оптимизации.


**Гипотеза:**


In [ ]:
# Решение пункта 2.2.


**Обсуждение результатов:**


### 2.3. Ширина сети и шаг обучения

На наборе `circles` исследуйте сеть с ReLU в двух последовательных сериях.
Используйте `seed=SEED`, `shuffle_seed=SHUFFLE_SEED`, `epochs=200`
и `batch_size=32`.

**Серия A.** Сравните ширины скрытого слоя `4, 16, 64` при `lr=0.1`.
Постройте зависимость конечных train/validation loss от ширины и выберите
ширину по validation.

**Серия B.** Для выбранной ширины сравните шаги `0.01, 0.1, 0.5`.
Начальные параметры и порядок батчей должны совпадать во всех трёх запусках.
Постройте кривые train/validation loss по эпохам и выберите шаг по validation.

Сведите настройки и конечные метрики двух серий в общую таблицу.
Всего в сериях пять различных конфигураций; совпадающий запуск из пункта 2.2
можно использовать повторно. Расходящиеся запуски отметьте отдельно;
при необходимости допустим один дополнительный диагностический запуск.

Обсудите влияние ширины и шага на результаты. Обоснуйте, позволяют ли
наблюдения судить о недообучении, переобучении и устойчивости оптимизации.


**Серия A. Гипотеза:**


In [ ]:
# Серия A.


**Серия B. Выбор ширины и гипотеза:**


In [ ]:
# Серия B.


**Обсуждение результатов и выбранные настройки:**


### 2.4. Функция активации и обучение скрытого слоя

Сравните ReLU с предоставленной пороговой активацией `Step`:

$$
\operatorname{Step}(x) =
\begin{cases}
1, & x > 0, \\
0, & x \leq 0.
\end{cases}
$$

В её обратном проходе используется нулевая производная; в точке разрыва
это соглашение реализации. Реализацию `Step` изменять нельзя.
Используйте `circles` и выбранные в пункте 2.3 ширину и шаг обучения.

**Диагностика одного обновления.** Создайте сети с ReLU и Step при `seed=SEED`.
Начальные параметры должны попарно совпадать по значениям, но храниться
в независимых массивах. Выполните по одному шагу SGD на первых 32 объектах
train в порядке хранения. Для каждой сети представьте loss до и после шага,
нормы градиентов `W1`, `b1`, `W2`, `b2` перед обновлением
и нормы изменения этих параметров после обновления.

**Полное обучение.** Обучите обе архитектуры заново для инициализаций
`42, 43, 44` — всего шесть запусков. В каждой паре начальные параметры
должны совпадать. Шаг, ширина, число эпох, размер и порядок батчей одинаковы
во всех запусках; отдельно настраивать Step не требуется.

Представьте:

- таблицу train/validation loss и accuracy всех шести запусков и норм
  изменения матрицы весов первого слоя;
- среднее и выборочное стандартное отклонение validation accuracy
  по трём инициализациям для каждой активации (`ddof=1`);
- кривые train/validation loss и разделяющие границы двух моделей
  с `seed=SEED`.

Объясните связь между градиентами, изменением параметров и качеством моделей.
Оцените чувствительность результатов к инициализации и сформулируйте
ограничения сравнения активаций при общем режиме обучения.


In [ ]:
class Step(Module):
    """Пороговая активация с нулевым градиентом в обратном проходе."""

    def forward(self, X: np.ndarray) -> np.ndarray:
        return (X > 0).astype(np.float64)

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        return np.zeros_like(grad_output)

**Гипотеза:**


In [ ]:
# Диагностика одного обновления.


In [ ]:
# Сравнение активаций по трём инициализациям.


**Обсуждение результатов:**


### 2.5. Итоговая оценка

По результатам пунктов 2.2–2.4 выберите одну из конфигураций сети со скрытым
слоем для `circles`. До обращения к test зафиксируйте архитектуру,
предобработку и режим обучения, обоснуйте выбор. Используйте сохранённую
модель с `seed=SEED`. Линейный классификатор из пункта 2.2 служит базовой моделью.

Оцените обе модели на test набора `circles`. Представьте таблицу из двух строк
с настройками, validation loss/accuracy выбранных запусков и test loss/accuracy.
После просмотра test конфигурации не меняйте; дообучение на train + validation
не предусмотрено. Тестовая часть `clouds` в работе не используется.

Разберите 3–5 ошибок выбранной сети: приведите координаты объектов, истинные
и предсказанные классы либо отметьте ошибки на диаграмме. Если ошибок меньше,
покажите все; если их нет, укажите это явно.

Сопоставьте итоговую оценку с выводами предыдущих экспериментов. Укажите
ограничение проведённого исследования и предложите следующий эксперимент,
не выполняя его.


**Выбранная конфигурация и обоснование:**


In [ ]:
# Итоговая оценка.


**Обсуждение результатов:**
